In [42]:
import pyspark
from pyspark.sql import SparkSession
from datetime import datetime as dt

## Question 2 read parquet, repartition into 4 parquet files and write to local directory

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [4]:
## repartition main parquet into 4 parquet files
df = df.repartition(4)

In [5]:
## Write repartitioned dataframe to local file folder
    ## will write as 4 parquet files
    ## set subdirectory to retain the repartitioned files
df.write.parquet('spark_hmwk/2025/11')

## Question 3: Count records
How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

In [11]:
df.show(2)

[Stage 13:==========================================>             (12 + 4) / 16]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [92]:
df.limit(2).toPandas()  ## see Spark DataFrame in friendlier Pandas presentation

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2025-11-07 15:04:17,2025-11-07 15:39:15,1,7.3,1,N,262,127,1,38.7,0.0,0.5,8.54,0.0,1.0,51.24,2.5,0.0,0.00
1,2,2025-11-08 17:51:05,2025-11-08 18:02:09,1,1.2,1,N,114,107,1,11.4,0.0,0.5,3.23,0.0,1.0,19.38,2.5,0.0,0.75


In [94]:
## Spark has to register PySpark DataFrame as a view or table before you can query it with SQL
df.createOrReplaceTempView("df")

## SELECT Query to add a date column from the timestamp column 'tpep_pickup_datetime'
df_date = spark.sql("""
    SELECT *,
    TO_DATE(tpep_pickup_datetime) AS pickup_date
    FROM df
""")

## save queried table as a view to use in downstream queries
df_date.createOrReplaceTempView("df_date")

In [95]:
df_date.limit(1).toPandas()  ## see format of datetime column "pickup_datetime"

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_date
0,2,2025-11-07 15:04:17,2025-11-07 15:39:15,1,7.3,1,N,262,127,1,...,0.0,0.5,8.54,0.0,1.0,51.24,2.5,0.0,0.0,2025-11-07


In [35]:
## filter by date and do groupby
    ## from statement is the View created with the datetime column created and saved
    
spark.sql("""
    SELECT pickup_date,
        COUNT(*)
    FROM df_date
    WHERE pickup_date == '2025-11-15'
    GROUP BY 1
""").show()

+-----------+--------+
|pickup_date|count(1)|
+-----------+--------+
| 2025-11-15|  162604|
+-----------+--------+



## Question 4

What is the length of the longest trip in the dataset in hours?

In [29]:
df = spark.read \
    .option("header", "true") \
    .parquet('spark_hmwk/2025/11')

In [6]:
df.show(2)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|              1|          7.3|         1|                 N|         262|    

### Test Pandas calculations to derive total hours in test dataframe

In [96]:
## see dataframe in user friendly "Pandas" view to plan out the queries to follow
    ## need to see columns and column types that are not as visible in spark.show() ascII presentation
dfx = df.limit(5).toPandas()

dfx

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2025-11-07 15:04:17,2025-11-07 15:39:15,1,7.30,1,N,262,127,1,38.7,0.0,0.5,8.54,0.0,1.0,51.24,2.5,0.0,0.00
1,2,2025-11-08 17:51:05,2025-11-08 18:02:09,1,1.20,1,N,114,107,1,11.4,0.0,0.5,3.23,0.0,1.0,19.38,2.5,0.0,0.75
2,7,2025-11-03 17:34:26,2025-11-03 17:34:26,1,5.06,1,N,162,209,1,24.7,0.0,0.5,6.39,0.0,1.0,38.34,2.5,0.0,0.75
3,2,2025-11-06 15:03:28,2025-11-06 15:27:07,1,3.33,1,N,163,24,1,23.3,0.0,0.5,4.00,0.0,1.0,32.05,2.5,0.0,0.75
4,2,2025-11-05 21:06:15,2025-11-05 21:12:03,1,0.90,1,N,189,49,1,7.2,1.0,0.5,1.94,0.0,1.0,11.64,0.0,0.0,0.00


In [97]:
dfx.dtypes   ## dropoff and pickup columns are datetime columns

VendorID                          int32
tpep_pickup_datetime     datetime64[ns]
tpep_dropoff_datetime    datetime64[ns]
passenger_count                   int64
trip_distance                   float64
RatecodeID                        int64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

In [98]:
## Calculate hours and minutes by calculating the time delta and then extracting the hours and the minutes and adding them together for a total hours and minutes per trip 
dfx['timedelta'] = dfx['tpep_dropoff_datetime'] - dfx['tpep_pickup_datetime']
dfx['day_hours'] = dfx['timedelta'].dt.components['days']*24
dfx['hours'] = dfx['timedelta'].dt.components['hours']
dfx['minutes_hours'] = dfx['timedelta'].dt.components['minutes']/60
dfx['total_hours'] = (dfx['day_hours'] + dfx.hours + dfx['minutes_hours'])
dfx

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,timedelta,day_hours,hours,minutes_hours,total_hours
0,2,2025-11-07 15:04:17,2025-11-07 15:39:15,1,7.30,1,N,262,127,1,...,1.0,51.24,2.5,0.0,0.00,0 days 00:34:58,0,0,0.566667,0.566667
1,2,2025-11-08 17:51:05,2025-11-08 18:02:09,1,1.20,1,N,114,107,1,...,1.0,19.38,2.5,0.0,0.75,0 days 00:11:04,0,0,0.183333,0.183333
2,7,2025-11-03 17:34:26,2025-11-03 17:34:26,1,5.06,1,N,162,209,1,...,1.0,38.34,2.5,0.0,0.75,0 days 00:00:00,0,0,0.000000,0.000000
3,2,2025-11-06 15:03:28,2025-11-06 15:27:07,1,3.33,1,N,163,24,1,...,1.0,32.05,2.5,0.0,0.75,0 days 00:23:39,0,0,0.383333,0.383333
4,2,2025-11-05 21:06:15,2025-11-05 21:12:03,1,0.90,1,N,189,49,1,...,1.0,11.64,0.0,0.0,0.00,0 days 00:05:48,0,0,0.083333,0.083333


## Process to take will be:

1. register and create a temporary view of Spark Dataframe "df"
2. query the temporary view "df" and create time delta column based on dropoff and pickup datetime columns. Order by the time delta column in DESC and save top 2 results into a Pandas Spark DataFrame

3. use datetime package in pandas to create hourly columns as extracted by the time delta column (days in hours, hours, and minutes/hours). Calculate total hours with the newly created hourly columns.

In [ ]:
## Needed packages for execution of queries and Pandas DataFrame creation
from datetime import datetime as dt

In [60]:
## register and create temporary view of Spark Dataframe "df" 
df.createOrReplaceTempView("df")

## Query the temporary view 
    ## create timedelta column between dropoff and pickup columns; filter on desc order on 'timedelta'
    ## save top 2 results of largest 'timedelta'
    ## save results into Pandas Spark Dataframe
df1 = spark.sql("""
    SELECT *,
        (tpep_dropoff_datetime - tpep_pickup_datetime) AS timedelta
    FROM df
    ORDER BY timedelta DESC
""").limit(2).toPandas()

## Extract datetime data from Panda Spark DataFrame and calculate total hours (days, hours, and minutes/hours)
df1['day_hours'] = df1['timedelta'].dt.components['days']*24
df1['hours'] = df1['timedelta'].dt.components['hours']
df1['minutes_hours'] = df1['timedelta'].dt.components['minutes']/60
df1['total_hours'] = (df1['day_hours'] + df1.hours + df1['minutes_hours'])


df1

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,timedelta,day_hours,hours,minutes_hours,total_hours
0,2,2025-11-26 20:22:12,2025-11-30 15:01:00,1,121.17,4,N,265,265,2,...,1.0,889.10,0.0,0.0,0.00,3 days 18:38:48,72,18,0.633333,90.633333
1,2,2025-11-27 04:22:41,2025-11-30 09:19:35,1,1.08,1,N,246,48,2,...,1.0,13.65,2.5,0.0,0.75,3 days 04:56:54,72,4,0.933333,76.933333


## Question 6

Load the zone lookup data into a temp view in Spark:

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [65]:
## pyspark packages needed to declare your schema on read
from pyspark.sql import types

In [64]:
## read in csv   
    ## NOTE!!! = Spark does not infer schema and will auto-assign String type as Schema unless schema is known like in a parquet file
    ## since this is a csv... schema is not known upon read
df_zone = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

## see head(2) view of Spark DataFrame as a Pandas DataFrame
    ## need to create structure with Schema to declare in Spark
df_zone.limit(2).toPandas()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone


In [67]:
## read data into Spark DataFrame again with declared schema
    ## schema created as structtype in VS Code
schema = types.StructType([
    types.StructField('LocationID', types.IntegerType(), True),
    types.StructField('Borough', types.StringType(), True),
    types.StructField('Zone', types.StringType(), True),
    types.StructField('service_zone', types.StringType(), True)
])

In [83]:
##read in csv and declare schema on read

df_zone = spark.read\
    .option("header", "true")\
    .schema(schema)\
    .csv('taxi_zone_lookup.csv')

## see sample of Spark DataFrame as a pandas dataframe
df_zone.limit(5).toPandas()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [82]:
df.limit(3).toPandas()   ##common column is PULocationID in "df" and LocationID in "df_zone"

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2025-11-07 15:04:17,2025-11-07 15:39:15,1,7.30,1,N,262,127,1,38.7,0.0,0.5,8.54,0.0,1.0,51.24,2.5,0.0,0.00
1,2,2025-11-08 17:51:05,2025-11-08 18:02:09,1,1.20,1,N,114,107,1,11.4,0.0,0.5,3.23,0.0,1.0,19.38,2.5,0.0,0.75
2,7,2025-11-03 17:34:26,2025-11-03 17:34:26,1,5.06,1,N,162,209,1,24.7,0.0,0.5,6.39,0.0,1.0,38.34,2.5,0.0,0.75


In [76]:
## verify df_zone has variety of LocationID
df_zone.count()     ##verify that Spark DataFrame is small


##verified.. now see array of all locationID

df_zone.toPandas().LocationID.unique()   ##confirmed variety in LocationID b

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

### Common Columns to JOIN on

1. df_zone (smaller table) has column "LocationID"

2. df (larger table) has column "PULocationID"

NOTE: will use Pandas API Syntax

In [78]:
## Needed pyspark package to broadcast smaller table during join
from pyspark.sql.functions import broadcast

In [85]:
df_merge = df.join(
    broadcast(df_zone), df.PULocationID == df_zone.LocationID, "left"
)

df_merge.limit(2).toPandas()    ## see sample in Pandas DataFrame

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,LocationID,Borough,Zone,service_zone
0,2,2025-11-07 15:04:17,2025-11-07 15:39:15,1,7.3,1,N,262,127,1,...,0.0,1.0,51.24,2.5,0.0,0.00,262,Manhattan,Yorkville East,Yellow Zone
1,2,2025-11-08 17:51:05,2025-11-08 18:02:09,1,1.2,1,N,114,107,1,...,0.0,1.0,19.38,2.5,0.0,0.75,114,Manhattan,Greenwich Village South,Yellow Zone


In [99]:
## register df_merge Spark DataFrame and then query to find least used zone for pickup

### ANSWER IS IN THIS OUTPUT

df_merge.createOrReplaceTempView("df_merge")

## use Spark SQL 
    ## need COUNT(*) on PULocationID column from larger Spark DataFrame
    ## ORDER BY ASCENDING
spark.sql("""
    SELECT Zone,
    COUNT(*) AS zone_count
    FROM df_merge
    GROUP BY 1
    ORDER BY zone_count
""").limit(3).toPandas()

,Zone,zone_count
0,Governor's Island/Ellis Island/Liberty Island,1
1,Arden Heights,1
2,Eltingville/Annadale/Prince's Bay,1
